# analysis.db 查询面板

用于查询旧版汇总库 `db/analysis.db`。

支持：
- `daily_metrics`
- `daily_size_breakdown`
- `daily_direction_breakdown`
- `daily_pd_breakdown`
- `daily_competitor_metrics`

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Hiragino Sans GB', 'Heiti SC', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

DB_PATH = '/Users/yy/.openclaw/workspace/db/analysis.db'
conn = sqlite3.connect(DB_PATH)
print('Connected:', DB_PATH)

In [ ]:
# 参数区
TARGET_DATE = None      # 例如 '2026-04-01'，None=全部
TARGET_CONTRACT = None  # 'k1'/'k2'/'k12'，None=全部

where = []
if TARGET_DATE:
    where.append(f"date = '{TARGET_DATE}'")
if TARGET_CONTRACT:
    where.append(f"contract = '{TARGET_CONTRACT}'")
WHERE_SQL = ('WHERE ' + ' AND '.join(where)) if where else ''
print('WHERE_SQL =', WHERE_SQL)

In [ ]:
# 0) 查看所有表
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
display(tables)

In [ ]:
# 1) 每日核心汇总 daily_metrics
sql_metrics = f"""
SELECT contract, date, total_orders, total_alpha, total_slippage, total_pl,
       total_bundle_fee, total_gas_fee, total_volume, slippage_per_order, core_factor
FROM daily_metrics
{WHERE_SQL}
ORDER BY date DESC, contract
"""
df_metrics = pd.read_sql_query(sql_metrics, conn)
display(df_metrics)

In [ ]:
# 2) 规模分层 daily_size_breakdown
sql_size = f"""
SELECT contract, date, size_bin, order_count, alpha_sum, alpha_mean, slippage_sum, slippage_mean, pl_mean
FROM daily_size_breakdown
{WHERE_SQL}
ORDER BY date DESC, contract, size_bin
"""
df_size = pd.read_sql_query(sql_size, conn)
display(df_size)

In [ ]:
# 3) 买卖方向 daily_direction_breakdown
sql_dir = f"""
SELECT contract, date, direction, order_count, alpha_sum, slippage_sum, pl_sum
FROM daily_direction_breakdown
{WHERE_SQL}
ORDER BY date DESC, contract, direction
"""
df_dir = pd.read_sql_query(sql_dir, conn)
display(df_dir)

In [ ]:
# 4) 价差分层 daily_pd_breakdown
sql_pd = f"""
SELECT contract, date, pd_bin, order_count, alpha_sum, alpha_mean, slippage_mean
FROM daily_pd_breakdown
{WHERE_SQL}
ORDER BY date DESC, contract, pd_bin
"""
df_pd = pd.read_sql_query(sql_pd, conn)
display(df_pd)

In [ ]:
# 5) 竞对汇总 daily_competitor_metrics
sql_comp = f"""
SELECT competitor, contract, date, total_orders, total_alpha, total_slippage, total_pl,
       total_bundle_fee, total_gas_fee, total_volume, slippage_per_order
FROM daily_competitor_metrics
{WHERE_SQL}
ORDER BY date DESC, competitor, contract
"""
df_comp = pd.read_sql_query(sql_comp, conn)
display(df_comp)

In [ ]:
# 可视化 A：核心指标趋势（total_alpha / total_slippage）
if 'df_metrics' in globals() and not df_metrics.empty:
    d = df_metrics.copy()
    d['date'] = pd.to_datetime(d['date'], errors='coerce')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.lineplot(data=d, x='date', y='total_alpha', hue='contract', marker='o', ax=axes[0])
    axes[0].set_title('total_alpha 趋势')
    axes[0].axhline(0, color='black', linewidth=0.8)

    sns.lineplot(data=d, x='date', y='total_slippage', hue='contract', marker='o', ax=axes[1])
    axes[1].set_title('total_slippage 趋势')
    axes[1].axhline(0, color='black', linewidth=0.8)

    plt.tight_layout()
    plt.show()
else:
    print('请先运行 daily_metrics 查询单元。')

In [ ]:
# 可视化 B：规模分层热力图（alpha_sum）
if 'df_size' in globals() and not df_size.empty:
    p = df_size.copy()
    p['contract_size'] = p['contract'] + ' | ' + p['size_bin']
    mat = p.pivot_table(index='size_bin', columns='contract', values='alpha_sum', aggfunc='sum')
    plt.figure(figsize=(8, 4))
    sns.heatmap(mat, annot=True, fmt='.1f', cmap='RdYlGn', center=0)
    plt.title('规模分层 alpha_sum 热力图')
    plt.tight_layout()
    plt.show()
else:
    print('请先运行 daily_size_breakdown 查询单元。')

In [ ]:
# 可选：关闭连接
# conn.close()